# Week 75, Day 4: ORMs and SQLAlchemy

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w075d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.executescript("""
CREATE TABLE authors (id INTEGER PRIMARY KEY, name TEXT);
CREATE TABLE books (id INTEGER PRIMARY KEY, author_id INTEGER, title TEXT);
INSERT INTO authors VALUES (1, 'Austen'), (2, 'Dickens'), (3, 'Eliot');
INSERT INTO books VALUES (1, 1, 'Emma'), (2, 1, 'Persuasion'), (3, 2, 'Bleak House'), (4, 3, 'Middlemarch');
""")
sent = []
conn.set_trace_callback(sent.append)

for author_id, name in conn.execute("SELECT id, name FROM authors").fetchall():
    titles = [t for (t,) in conn.execute("SELECT title FROM books WHERE author_id = ?", (author_id,))]
    print(name, titles)
print(len(sent), "queries")

In [ ]:
import sqlite3

TYPES = {int: "INTEGER", str: "TEXT", float: "REAL"}

class Field:
    def __init__(self, kind, required=True):
        self.kind, self.required = kind, required
    def __set_name__(self, owner, name):
        self.name = name
    def __get__(self, obj, owner=None):
        return self if obj is None else obj.__dict__.get(self.name)
    def __set__(self, obj, value):
        if value is not None and not isinstance(value, self.kind):
            raise TypeError(f"{self.name} must be {self.kind.__name__}")
        obj.__dict__[self.name] = value

class Model:
    def __init_subclass__(cls, table, **kwargs):
        super().__init_subclass__(**kwargs)
        cls.table = table
        cls.fields = [v for v in vars(cls).values() if isinstance(v, Field)]

    def __init__(self, id=None, **values):
        self.id = id
        for f in self.fields:
            setattr(self, f.name, values.get(f.name))

    @classmethod
    def create_sql(cls):
        columns = ["id INTEGER PRIMARY KEY"] + [f"{f.name} {TYPES[f.kind]}{' NOT NULL' if f.required else ''}" for f in cls.fields]
        return f"CREATE TABLE {cls.table} ({', '.join(columns)})"

    def save(self, conn):
        names = [f.name for f in self.fields]
        values = [getattr(self, n) for n in names]
        with conn:
            if self.id is None:
                cur = conn.execute(f"INSERT INTO {self.table} ({', '.join(names)}) VALUES ({', '.join('?' * len(names))})", values)
                self.id = cur.lastrowid
            else:
                conn.execute(f"UPDATE {self.table} SET {', '.join(n + ' = ?' for n in names)} WHERE id = ?", values + [self.id])

class Book(Model, table="books"):
    title = Field(str)
    year = Field(int)
    rating = Field(float, required=False)

conn = sqlite3.connect(":memory:")
print(Book.create_sql())
conn.execute(Book.create_sql())
emma = Book(title="Emma", year=1815)
emma.save(conn)
emma.rating = 4.5
emma.save(conn)
print(emma.id, conn.execute("SELECT * FROM books").fetchall())

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.executescript("""
CREATE TABLE authors (id INTEGER PRIMARY KEY, name TEXT);
CREATE TABLE books (id INTEGER PRIMARY KEY, author_id INTEGER, title TEXT);
INSERT INTO authors VALUES (1, 'Austen'), (2, 'Dickens'), (3, 'Eliot');
INSERT INTO books VALUES (1, 1, 'Emma'), (2, 1, 'Persuasion'), (3, 2, 'Bleak House'), (4, 3, 'Middlemarch');
""")
sent = []
conn.set_trace_callback(sent.append)

authors = conn.execute("SELECT id, name FROM authors ORDER BY id").fetchall()
ids = [a[0] for a in authors]
books = {}
for author_id, title in conn.execute(f"SELECT author_id, title FROM books WHERE author_id IN ({', '.join('?' * len(ids))}) ORDER BY id", ids):
    books.setdefault(author_id, []).append(title)
for author_id, name in authors:
    print(name, books.get(author_id, []))
print(len(sent), "queries, however many authors there are")

> **Key idea.** An ORM maps classes to tables, attributes to columns and objects to rows; it saves and loads objects, keeps an identity map (one object per row in a session), tracks changes as a unit of work, follows relationships lazily or eagerly, and builds queries from Python expressions. Under the hood: descriptors for columns, class hooks to collect them, parameters for values and fixed names for identifiers. Lazy relationships in a loop cause the N+1 problem; load related rows eagerly with a join or one IN query, and watch the SQL that's sent (set_trace_callback, or echo=True). SQLAlchemy 2.0 declares models with Mapped and mapped_column, queries with select(), and migrates with Alembic.

### Exercise: A table from a class

Write `Field(kind, required=True, unique=False)` (a descriptor that type-checks values on assignment, raising `TypeError`, and stores them in the instance's `__dict__`) and `Model`, whose `__init_subclass__(cls, table)` records `cls.table` and the class's fields **in definition order** in `cls.fields`. Then `create_sql(model)`: `CREATE TABLE table (id INTEGER PRIMARY KEY, name TYPE [NOT NULL] [UNIQUE], ...)` with `int` as `INTEGER`, `str` as `TEXT`, `float` as `REAL` and `bool` as `INTEGER`. Field names must be plain identifiers (`ValueError` otherwise, checked in `__set_name__`).

In [ ]:
import re

TYPES = {int: "INTEGER", str: "TEXT", float: "REAL", bool: "INTEGER"}

class Field:
    ___

class Model:
    ___

def create_sql(model):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_schema():\n    \"classes to tables\"\n    import sqlite3\n    class Book(Model, table=\"books\"):\n        title = Field(str)\n        isbn = Field(str, unique=True)\n        year = Field(int)\n        rating = Field(float, required=False)\n        lent = Field(bool)\n    assert Book.table == \"books\" and [f.name for f in Book.fields] == [\"title\", \"isbn\", \"year\", \"rating\", \"lent\"]\n    assert create_sql(Book) == \"CREATE TABLE books (id INTEGER PRIMARY KEY, title TEXT NOT NULL, isbn TEXT NOT NULL UNIQUE, year INTEGER NOT NULL, rating REAL, lent INTEGER NOT NULL)\"\n    sqlite3.connect(\":memory:\").execute(create_sql(Book))\n    b = Book()\n    b.year = 1815\n    try:\n        b.year = \"1815\"\n        assert False\n    except TypeError:\n        pass\n    assert b.year == 1815 and Book.year.name == \"year\"")

### Exercise: Saving and loading

Using the given `Field` and `Model`, add class methods and methods to `Model`: `obj.save(conn)` (an `INSERT` setting `obj.id` if it has none, otherwise an `UPDATE` of every field by id; committed), `Model.get(conn, id)` (an instance, or `None`), `Model.find(conn, **equals)` (instances matching every `field=value` given, by id; unknown field names raise `ValueError`), and `obj.delete(conn)` (deleting the row and setting `obj.id` to `None`). Values are always parameters.

In [ ]:
import sqlite3

class Field:
    def __init__(self, kind):
        self.kind = kind
    def __set_name__(self, owner, name):
        self.name = name
    def __get__(self, obj, owner=None):
        return self if obj is None else obj.__dict__.get(self.name)
    def __set__(self, obj, value):
        if value is not None and not isinstance(value, self.kind):
            raise TypeError(self.name)
        obj.__dict__[self.name] = value

class Model:
    def __init_subclass__(cls, table=None, **kwargs):
        super().__init_subclass__(**kwargs)
        cls.table = table
        cls.fields = [v for v in vars(cls).values() if isinstance(v, Field)]

    def __init__(self, id=None, **values):
        self.id = id
        for f in self.fields:
            setattr(self, f.name, values.get(f.name))

    def save(self, conn):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_crud():\n    \"insert, update, get, find and delete\"\n    import sqlite3\n    class Book(Model, table=\"books\"):\n        title = Field(str)\n        year = Field(int)\n    conn = sqlite3.connect(\":memory:\")\n    conn.execute(\"CREATE TABLE books (id INTEGER PRIMARY KEY, title TEXT, year INTEGER)\")\n    emma, persuasion = Book(title=\"Emma\", year=1815), Book(title=\"Persuasion\", year=1817)\n    emma.save(conn)\n    persuasion.save(conn)\n    assert (emma.id, persuasion.id) == (1, 2) and not conn.in_transaction\n    emma.year = 1816\n    emma.save(conn)\n    loaded = Book.get(conn, 1)\n    assert (loaded.id, loaded.title, loaded.year) == (1, \"Emma\", 1816) and Book.get(conn, 9) is None\n    assert [b.title for b in Book.find(conn, year=1817)] == [\"Persuasion\"] and len(Book.find(conn)) == 2\n    assert Book.find(conn, title=\"x' OR '1'='1\") == []\n    try:\n        Book.find(conn, author=\"x\")\n        assert False\n    except ValueError:\n        pass\n    persuasion.delete(conn)\n    assert persuasion.id is None and [b.id for b in Book.find(conn)] == [1]")

### Exercise: Counting the queries

Write `count_queries(conn, func)`: run `func()` while recording every statement the connection sends (with `conn.set_trace_callback`), and return `(result, statements)`, removing the callback afterwards even if `func` raises. Then `books_by_author(conn)`: a dict of author name to the list of their book titles (in book id order; authors with no books get `[]`), using **at most two** queries however many authors there are.

In [ ]:
import sqlite3

def count_queries(conn, func):
    ___

def books_by_author(conn):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_queries():\n    \"no N+1\"\n    import sqlite3\n    conn = sqlite3.connect(\":memory:\")\n    conn.executescript(\"\"\"\n    CREATE TABLE authors (id INTEGER PRIMARY KEY, name TEXT);\n    CREATE TABLE books (id INTEGER PRIMARY KEY, author_id INTEGER, title TEXT);\n    INSERT INTO authors VALUES (1, 'Austen'), (2, 'Dickens'), (3, 'Eliot'), (4, 'Woolf');\n    INSERT INTO books VALUES (1, 1, 'Emma'), (2, 2, 'Bleak House'), (3, 1, 'Persuasion'), (4, 3, 'Middlemarch');\n    \"\"\")\n    result, statements = count_queries(conn, lambda: books_by_author(conn))\n    assert result == {\"Austen\": [\"Emma\", \"Persuasion\"], \"Dickens\": [\"Bleak House\"], \"Eliot\": [\"Middlemarch\"], \"Woolf\": []}\n    assert len(statements) <= 2, statements\n    conn.executemany(\"INSERT INTO authors (name) VALUES (?)\", [(f\"A{i}\",) for i in range(50)])\n    _, statements = count_queries(conn, lambda: books_by_author(conn))\n    assert len(statements) <= 2\n    conn.execute(\"SELECT 1\")\n    try:\n        count_queries(conn, lambda: 1 / 0)\n    except ZeroDivisionError:\n        pass\n    _, after = count_queries(conn, lambda: None)\n    assert after == []")

### Exercise: An identity map

Write a `Session(conn)` for the given `Book` model: `session.get(Book, id)` loads a row once and returns the **same** object for later calls with the same class and id (or `None` if there's no row, not cached); `session.dirty()` lists the loaded objects (in load order) whose field values differ from what was loaded; and `session.commit()` writes an `UPDATE` (of only the changed fields) for each dirty object in one transaction, after which nothing is dirty. Keep, for each loaded object, a copy of its values as loaded.

In [ ]:
import sqlite3

class Book:
    table, fields = "books", ["title", "year"]
    def __init__(self, id, title, year):
        self.id, self.title, self.year = id, title, year

class Session:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_identity():\n    \"one object per row, and only changed rows written\"\n    import sqlite3\n    conn = sqlite3.connect(\":memory:\")\n    conn.executescript(\"CREATE TABLE books (id INTEGER PRIMARY KEY, title TEXT, year INTEGER); INSERT INTO books VALUES (1, 'Emma', 1815), (2, 'Persuasion', 1817);\")\n    s = Session(conn)\n    a, b = s.get(Book, 1), s.get(Book, 1)\n    assert a is b and a.title == \"Emma\" and s.get(Book, 9) is None\n    p = s.get(Book, 2)\n    assert s.dirty() == []\n    a.year = 1816\n    assert s.dirty() == [a]\n    sent = []\n    conn.set_trace_callback(sent.append)\n    s.commit()\n    conn.set_trace_callback(None)\n    updates = [q for q in sent if q.startswith(\"UPDATE\")]\n    assert len(updates) == 1 and \"year\" in updates[0] and \"title\" not in updates[0], sent\n    assert conn.execute(\"SELECT year FROM books WHERE id = 1\").fetchone() == (1816,) and s.dirty() == []\n    assert Session(conn).get(Book, 1) is not a")

### Exercise: Queries from Python expressions

Build the core of an expression language like SQLAlchemy's. `Column(name)` objects overload `==`, `!=`, `<`, `<=`, `>` and `>=` to return a `Condition`, and have a `.like(pattern)` method. A `Condition` has `.sql` (text with `?` placeholders) and `.params` (a list), and combines with `&` (AND) and `|` (OR), bracketed: `(A) AND (B)`. Comparing with `None` using `==` or `!=` gives `name IS NULL` / `name IS NOT NULL` with no parameter. Then `select(table, *columns)` returns a query object with `.where(condition)` (several calls are ANDed), `.order_by(column, descending=False)`, and `.compile()` giving `(sql, params)`.

In [ ]:
class Condition:
    ___

class Column:
    ___

class Select:
    ___

def select(table, *columns):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_expressions():\n    \"operator overloading builds SQL\"\n    import sqlite3\n    title, year, rating = Column(\"title\"), Column(\"year\"), Column(\"rating\")\n    c = (year > 1810) & ((title.like(\"%e%\")) | (rating == None))\n    assert c.sql == \"(year > ?) AND ((title LIKE ?) OR (rating IS NULL))\" and c.params == [1810, \"%e%\"]\n    q = select(\"books\", title, year).where(year >= 1815).where(rating != None).order_by(year, descending=True)\n    assert q.compile() == (\"SELECT title, year FROM books WHERE (year >= ?) AND (rating IS NOT NULL) ORDER BY year DESC\", [1815])\n    conn = sqlite3.connect(\":memory:\")\n    conn.executescript(\"CREATE TABLE books (title TEXT, year INTEGER, rating REAL); INSERT INTO books VALUES ('Emma', 1815, 4.5), ('Persuasion', 1817, NULL), ('Pamela', 1740, 3.0);\")\n    assert conn.execute(*q.compile()).fetchall() == [(\"Emma\", 1815)]\n    q2 = select(\"books\", title).where((year < 1800) | (title == \"Persuasion\")).order_by(title)\n    assert conn.execute(*q2.compile()).fetchall() == [(\"Pamela\",), (\"Persuasion\",)]\n    assert select(\"books\", title).compile() == (\"SELECT title FROM books\", [])")

### Bug hunt: A query per author

The loop sends one query **per author** for their books (N+1), so with these four authors the program sends five queries and prints `queries: 5`. Load all the books in one query and group them in Python, so it prints `queries: 2`, with the same results.

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.executescript("""
CREATE TABLE authors (id INTEGER PRIMARY KEY, name TEXT);
CREATE TABLE books (id INTEGER PRIMARY KEY, author_id INTEGER, title TEXT);
INSERT INTO authors VALUES (1, 'Austen'), (2, 'Dickens'), (3, 'Eliot'), (4, 'Woolf');
INSERT INTO books VALUES (1, 1, 'Emma'), (2, 2, 'Bleak House'), (3, 1, 'Persuasion'), (4, 3, 'Middlemarch');
""")
sent = []
conn.set_trace_callback(sent.append)

result = {}
for author_id, name in conn.execute("SELECT id, name FROM authors ORDER BY id").fetchall():
    result[name] = [t for (t,) in conn.execute("SELECT title FROM books WHERE author_id = ? ORDER BY id", (author_id,))]

conn.set_trace_callback(None)
print(result)
print("queries:", len(sent))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"two queries\"\n    expect_output(\"{'Austen': ['Emma', 'Persuasion'], 'Dickens': ['Bleak House'], 'Eliot': ['Middlemarch'], 'Woolf': []}\\nqueries: 2\")")

### Bug hunt: An update for every row

`save` builds its `UPDATE` without a `WHERE` clause, so changing one book's year changes **every** book's, and this prints `[(1, 'Emma', 1816), (2, 'Persuasion', 1816)]`. Update only the object's own row, so it prints `[(1, 'Emma', 1816), (2, 'Persuasion', 1817)]`.

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.executescript("CREATE TABLE books (id INTEGER PRIMARY KEY, title TEXT, year INTEGER); INSERT INTO books VALUES (1, 'Emma', 1815), (2, 'Persuasion', 1817);")

def save(conn, book_id, title, year):
    with conn:
        conn.execute("UPDATE books SET title = ?, year = ?", (title, year))

save(conn, 1, "Emma", 1816)
print(conn.execute("SELECT * FROM books ORDER BY id").fetchall())

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"only one row\"\n    expect_output(\"[(1, 'Emma', 1816), (2, 'Persuasion', 1817)]\")")

### Bug hunt: An identity map that mixed up tables

The cache is keyed by the row's id alone, so after loading author 1, asking for **book** 1 returns the author, and this prints `Author Austen`. Key the cache by the class **and** the id, so it prints `Book Emma`.

In [ ]:
class Author:
    def __init__(self, name):
        self.name = name

class Book:
    def __init__(self, name):
        self.name = name

rows = {(Author, 1): "Austen", (Book, 1): "Emma"}
cache = {}

def get(cls, id):
    if id not in cache:
        cache[id] = cls(rows[(cls, id)])
    return cache[id]

get(Author, 1)
found = get(Book, 1)
print(type(found).__name__, found.name)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"class and id\"\n    expect_output(\"Book Emma\")")

### Puzzle: A SQLAlchemy model

Arrange the lines of a SQLAlchemy 2.0 model for a table of books with a title and a year. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # queries to list 100 authors and lazily load each one's books in a loop
q = ___   # the same, loading all the books with one IN query
r = ___   # True or False: in a session with an identity map, two gets of the same row return the same object
s = ___   # the SQLAlchemy create_engine argument that prints every SQL statement
t = ___   # SQLAlchemy's migration tool

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 101 and q == 2 and r is True, \"p, q and r\"\n    assert s in (\"echo\", \"echo=True\") and t == \"Alembic\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: lazy query sets

Write `QuerySet(conn, table, columns)`, a lazy, chainable query in the style of Django's ORM: `filter(**conditions)` (with lookups as in Day 1's builder: `field`, `field__lt`, `__le`, `__gt`, `__ge`, `__ne`, `__like`, `__in`) and `order_by(*fields)` (with `-` for descending) each return a **new** query set and run nothing; iterating runs the query **once** and caches the rows (as dicts), so iterating again sends no query; `count()` sends a `SELECT count(*)` (or uses the cache if the query has already run); `first()` returns the first row (with `LIMIT 1` if not yet cached) or `None`; and `[i]` indexing uses the cache (running the query first if needed). Field names must be among `columns` (`ValueError` otherwise).

In [ ]:
import sqlite3

OPS = {"": "=", "lt": "<", "le": "<=", "gt": ">", "ge": ">=", "ne": "<>", "like": "LIKE"}

class QuerySet:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_querysets():\n    \"lazy, chainable and cached\"\n    import sqlite3\n    conn = sqlite3.connect(\":memory:\")\n    conn.executescript(\"CREATE TABLE books (id INTEGER PRIMARY KEY, title TEXT, year INTEGER); INSERT INTO books (title, year) VALUES ('Emma', 1815), ('Persuasion', 1817), ('Pamela', 1740), ('Middlemarch', 1871);\")\n    sent = []\n    conn.set_trace_callback(sent.append)\n    books = QuerySet(conn, \"books\", [\"id\", \"title\", \"year\"])\n    recent = books.filter(year__gt=1800).order_by(\"-year\")\n    assert sent == [], \"building a query set runs nothing\"\n    assert [b[\"title\"] for b in recent] == [\"Middlemarch\", \"Persuasion\", \"Emma\"] and len(sent) == 1\n    assert [b[\"year\"] for b in recent] == [1871, 1817, 1815] and len(sent) == 1, \"the second loop uses the cache\"\n    assert recent.count() == 3 and len(sent) == 1\n    fresh = books.filter(title__like=\"P%\")\n    assert fresh.count() == 2 and len(sent) == 2 and \"count(*)\" in sent[-1]\n    assert fresh.order_by(\"title\").first() == {\"id\": 3, \"title\": \"Pamela\", \"year\": 1740} and \"LIMIT 1\" in sent[-1]\n    assert books.filter(id__in=[2, 4]).order_by(\"id\")[1][\"title\"] == \"Middlemarch\"\n    assert books.filter(year=1700).first() is None\n    assert books.filter(year__gt=1800) is not books and [b[\"id\"] for b in books.order_by(\"id\")] == [1, 2, 3, 4]\n    try:\n        books.filter(author=\"x\")\n        assert False\n    except ValueError:\n        pass")

## Warm-up

One query for the authors, then **one more per author** for their books: with three authors that's four queries; with a thousand it's a thousand and one, each a trip to the database. This is the **N+1 problem**, and it's the easiest mistake to make with an ORM, because there the second query hides behind an innocent-looking `author.books`. `set_trace_callback` shows every statement a connection really sends: your best tool for catching it.

## What an ORM does

An **object-relational mapper** connects two worlds: tables, rows and foreign keys on one side; classes, objects and references on the other. It typically gives you:

- **Mapping**: a class per table, an attribute per column, an object per row. The ORM can create the tables from the classes.
- **Persistence**: `session.add(book)` and `session.commit()` write the right `INSERT` or `UPDATE`; `session.get(Book, 4)` loads one.
- **An identity map**: within a session, a row is loaded into **one** object, so two lookups of book 4 give the same Python object.
- **A unit of work**: the session tracks which objects are new, changed or deleted, and writes them all in one transaction at commit.
- **Relationships**: `book.author` and `author.books` follow foreign keys, loading related rows **lazily** (when first used) or **eagerly** (in the same query, with a join).
- **Queries in Python**: `select(Book).where(Book.year > 1850).order_by(Book.title)`, built from objects and turned into SQL with parameters.

What it costs: another layer to learn, SQL you didn't write (sometimes inefficient, as in the warm-up), and queries that are harder to see in the code. Good practice is to use the ORM for everyday work, to **look at the SQL it sends** (SQLAlchemy's `echo=True`), and to write SQL by hand for complex reports, where it's clearer.

## Classes as tables

At the heart of an ORM is a **class per table**, with a **descriptor** (Week 63) per column. A field descriptor knows its column's name and type; the base class's `__init_subclass__` collects the fields of each model class; and from those the ORM can write `CREATE TABLE`, `INSERT` and `SELECT` statements, always with parameters for the values and the class's own field names (never user input) for the identifiers.

## Sessions, identity and laziness

Real ORMs add a **session** between your objects and the connection:

- **Identity map.** The session keeps a dict from `(class, id)` to the loaded object. Loading the same row again returns the **same object**, so changes made through one reference are seen through every other, and the session knows which object to save.
- **Change tracking.** The session (or the objects themselves) records which attributes changed since loading, so it can `UPDATE` only those, and skip unchanged objects entirely.
- **Lazy loading.** `book.author` isn't loaded until it's used, and then it's one query (or none, if the identity map already has that author). That's efficient when you need related objects occasionally, and the cause of N+1 when you need them in a loop.
- **Eager loading.** When you know you'll need them, load related objects **with** the main query: a join (`joinedload` in SQLAlchemy), or one extra query for all of them at once (`selectinload`: `SELECT ... WHERE author_id IN (?, ?, ?)`). Two queries instead of N+1.

## SQLAlchemy

SQLAlchemy has two layers: **Core**, a Python way of building SQL expressions, and the **ORM** on top. In its 2.0 style, models are classes with typed attributes:

```python
from sqlalchemy import ForeignKey, String, create_engine, select
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column, relationship, selectinload

class Base(DeclarativeBase):
    pass

class Author(Base):
    __tablename__ = "authors"
    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))
    books: Mapped[list["Book"]] = relationship(back_populates="author")

class Book(Base):
    __tablename__ = "books"
    id: Mapped[int] = mapped_column(primary_key=True)
    title: Mapped[str]
    year: Mapped[int]
    author_id: Mapped[int] = mapped_column(ForeignKey("authors.id"))
    author: Mapped[Author] = relationship(back_populates="books")

engine = create_engine("sqlite:///library.db", echo=True)      # echo prints every SQL statement
Base.metadata.create_all(engine)

with Session(engine) as session:
    austen = Author(name="Austen", books=[Book(title="Emma", year=1815)])
    session.add(austen)
    session.commit()
    query = select(Author).options(selectinload(Author.books)).where(Author.name.like("A%"))
    for author in session.scalars(query):
        print(author.name, [b.title for b in author.books])     # no N+1: books were loaded with the authors
```

`Author.name.like("A%")` isn't a boolean: SQLAlchemy's column objects overload operators (`==`, `<`, `&`, `|`) to build **expression objects** that become SQL with parameters. Schema changes over time are handled by **Alembic**, SQLAlchemy's migration tool, which generates numbered migration scripts like yesterday's challenge.

## Practice

## On your own computer

Install SQLAlchemy and run the model from today's lesson, with `echo=True`, and read the SQL it sends:

```bash
pip install sqlalchemy
```

Then try removing `.options(selectinload(Author.books))` and adding a few more authors: you'll see the N+1 queries appear in the log, one `SELECT ... FROM books WHERE ? = books.author_id` per author. For Django projects, the equivalent is `prefetch_related("books")`, and the `django-debug-toolbar` shows every query a page sends.

## Quiz

## Daily challenge

## Before you go

If you have SQLAlchemy on your own computer, rebuild today's bookshop in it and watch the SQL with `echo=True`. Tomorrow: **transactions and indexes**: what the database promises when several things happen at once, and how it finds one row among millions without reading them all.